# ARE-004.1 - Prepared training data exploration
Only local ARE-004 Parquet is read; source datasets are NEVER downloaded here.
Install `pip install -e ".[data,research,dev]"`. Run from the project root/notebooks.
Artifacts are checked against the successful ARE-004 quality report.

The optional profile section makes two read-only requests for one owned profile.
No personal history is written. Clear all outputs before saving/sharing this file.


In [ ]:
from IPython.display import display
from anime_recommender.exploration import project_root
from anime_recommender.ingestion.processed_data import load_processed_data
from anime_recommender.evaluation.training_data import training_statistics, map_user_to_training

root = project_root()
processed_dir = root / "data/processed"
check_profile = True


## Load and inspect certified local tables

In [ ]:
data = load_processed_data(processed_dir)
interactions = data.interactions
stats = training_statistics(interactions)
display(interactions.head(10))
display(data.catalog.head())
display(interactions.dtypes)
display(stats["summary"])


## Ratings and activity
Zeros are preserved as raw ratings, not implicit dislikes.
Sparsity uses distinct user-item pairs and the active users/items of this sample;
it is not the sparsity of the full source dataset.


In [ ]:
display(stats["ratings"])
display(stats["per_user"].describe())
display(stats["per_anime"].describe())


## Match one profile through MAL ID
Internal dataset_anime_id differs from MAL ID. Catalog coverage and presence in
clean training interactions are separate. Presence in this sample does not prove
that an item will survive future training filters. No full source download occurs.


In [ ]:
if check_profile:
    import os
    import pandas as pd
    from dotenv import load_dotenv
    from anime_recommender.exploration import explore_profile
    from anime_recommender.ingestion.yummyanime import YummyAnimeClient

    load_dotenv(root / ".env", override=False)
    nickname = os.getenv("YUMMYANIME_TEST_NICKNAME", "").strip() or "First_Light"
    client = YummyAnimeClient(os.environ.get("YUMMYANIME_APPLICATION_TOKEN", ""))
    snapshot = explore_profile(client, nickname, cross_checks=False)
    display(pd.DataFrame(client.request_log))
    print("Profile available:", snapshot.complete, "Error:", snapshot.error)
    if snapshot.complete:
        matched, coverage = map_user_to_training(snapshot.frame, data.mapping, interactions)
        display(coverage)


## Conclusions and limitations
The cleaning policy removes exact duplicates and all conflicting rating pairs.
Local outputs and their quality report remain in data/ and are ignored by Git.
One profile is not representative of all YummyAnime users. Mapping coverage is not
recommendation quality. No model is trained or loaded. Restore manual parameters
and clear outputs before saving. A missing/mismatched local artifact causes a clear
error instead of silently downloading or regenerating data.
